# Tutorial 2: Types of Optimization Problems

**Prerequisites:** Tutorial 1 — What Is an Optimization Problem?  
**Up Next:** Tutorial 3 — Gradient Descent from Scratch

---

## Concept

Not all optimization problems are created equal. The structure of the objective function and constraints determines which solvers will work and how hard the problem is. Three key distinctions are:

- **Linear vs. nonlinear:** Is the objective (and are the constraints) a linear function of $\mathbf{x}$?
- **Constrained vs. unconstrained:** Must the solution satisfy additional inequalities or equalities?
- **Convex vs. non-convex:** Does the feasible objective have a single basin, or multiple local minima?

Linear, unconstrained, convex problems are the easiest. Our four-bar linkage problem is nonlinear, constrained, and non-convex — one of the hardest combinations.

## Key Takeaway

> **The four-bar path-synthesis problem is nonlinear (trigonometric FK), constrained (Grashof condition), and non-convex (multiple local minima). Recognizing these properties guides your choice of solver.**

## Math Core

| Property | Definition | Four-bar example |
|----------|-----------|------------------|
| **Linear** | $f(\mathbf{x}) = \mathbf{c}^T \mathbf{x}$, constraints $A\mathbf{x} \le \mathbf{b}$ | No — $f$ involves $\cos$, $\arccos$, $\sqrt{\cdot}$ |
| **Constrained** | $\exists\; g_i(\mathbf{x}) \le 0$ | Yes — Grashof: $\ell_1 + \max(\ell_0,\ell_2,\ell_3) \le \ell_0 + \ell_2 + \ell_3 - \ell_1$ and assembly |
| **Convex** | $f(\alpha\mathbf{x} + (1-\alpha)\mathbf{y}) \le \alpha f(\mathbf{x}) + (1-\alpha)f(\mathbf{y})$ | No — landscape has multiple valleys |

A function is **convex** if a line segment between any two points on its graph never dips below the graph.  When this fails, gradient-based methods can get trapped in local minima.

## Code

Let's set up the same four-bar objective from Tutorial 1 and then visualize these three properties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dms.curves import CompareCurves
%matplotlib inline

# Constants (same as Tutorial 1)
L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective: distance from coupler path to targets."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Nonlinearity

A linear function produces straight contour lines (hyperplanes).  Let's look at a 1-D slice through our objective to see it is far from linear.

In [ ]:
l2_slice = np.linspace(0.5, 2.5, 200)
f_slice = [objective([l2, 1.5]) for l2 in l2_slice]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(l2_slice, f_slice, 'b-', lw=2)
ax.set_xlabel(r'$\ell_2$ (coupler length)')
ax.set_ylabel(r'$f(\ell_2, 1.5)$')
ax.set_title('1-D slice: clearly nonlinear')
ax.set_ylim(0, 5)
plt.tight_layout()

### Constrained vs. unconstrained: the Grashof condition

A four-bar linkage can only make a full crank rotation if the **Grashof condition** holds: the shortest link plus the longest link must be no greater than the sum of the other two. When this fails (or when the loop cannot close), the mechanism is infeasible.

We already encode this: `objective` returns a large penalty (1000) when `fourbar_fk` returns `None`. Let's visualize the feasible region on the contour plot.

In [ ]:
l2_vals = np.linspace(0.3, 2.5, 80)
l3_vals = np.linspace(0.3, 2.5, 80)
L2, L3 = np.meshgrid(l2_vals, l3_vals)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2, L3)

fig, ax = plt.subplots(figsize=(7, 5))
feasible = Z < 999
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
ax.contour(L2, L3, feasible.astype(float), levels=[0.5],
           colors='red', linewidths=2, linestyles='--')
ax.set_xlabel(r'$\ell_2$ (coupler length)')
ax.set_ylabel(r'$\ell_3$ (follower length)')
ax.set_title('Feasible region (inside red boundary)')
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

The dark region outside the red boundary is infeasible — the mechanism cannot assemble for all required crank angles. Any optimizer we use must either respect these constraints directly or handle the penalty gracefully.

### Non-convexity: multiple local minima

A convex function has a single global minimum — any local minimum is also the global one. For a non-convex function, the midpoint of two low-cost points can have *higher* cost.

Let's demonstrate this on our landscape.

In [ ]:
# Two feasible points with low cost but high midpoint cost
xA = np.array([1.35, 2.3])
xB = np.array([1.4, 0.5])
xM = 0.5 * (xA + xB)  # midpoint

fA, fB, fM = objective(xA), objective(xB), objective(xM)
print(f'f(A) = {fA:.3f},  f(B) = {fB:.3f},  f(midpoint) = {fM:.3f}')
print(f'Convexity requires f(M) <= {0.5*fA + 0.5*fB:.3f}  '
      f'— {"satisfied" if fM <= 0.5*fA+0.5*fB else "VIOLATED"}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
ax.contour(L2, L3, feasible.astype(float), levels=[0.5],
           colors='red', linewidths=2, linestyles='--')

# Plot A, B, and midpoint
for pt, label, color in [(xA, 'A', 'cyan'), (xB, 'B', 'lime'),
                          (xM, 'M', 'red')]:
    ax.plot(*pt, 'o', color=color, ms=10, markeredgecolor='k')
    ax.annotate(f'{label}: f={objective(pt):.2f}',
                pt, textcoords='offset points',
                xytext=(8, 8), color='white', fontsize=9)
ax.plot([xA[0], xB[0]], [xA[1], xB[1]], 'w--', lw=1)
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Non-convexity: midpoint M has higher cost than A or B')
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

## Mechanism Hook

Let's visualize the mechanisms at points A and B to see that two very different designs can both be decent solutions — a hallmark of non-convex problems.

In [ ]:
from dms.mechanisms.fourbar import FourBar

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (label, x) in zip(axes, [('A', xA), ('B', xB)]):
    l2, l3 = x
    mechanism = FourBar(L0, L1, l2, l3)
    pts = [coupler_point(t, l2, l3) for t in THETAS]
    path = np.array([p for p in pts if p is not None])
    mechanism.plot(np.deg2rad(90), ax=ax)
    ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
    if len(path) > 0:
        ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.set_title(f'{label}: $\\ell_2$={l2}, $\\ell_3$={l3}, '
                 f'f={objective(x):.3f}')
plt.tight_layout()

Two different linkage geometries, two different local minima. A gradient-based solver starting near A would never discover B, and vice versa. This is why non-convex problems are hard — and why later tutorials explore global methods like simulated annealing and genetic algorithms.

---